# Fluid Propagation

This test is meant to demonstrate the Fluid Propagation algorithm

In [ ]:
import gstlearn as gl
import gstlearn.plot as gp
import gstlearn.document as gdoc

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

## Data Presentation

The block system is constituted of 50 by 50 cells. Each cell measures 1m by 1m. Therefore, the grid lies between 1m and 50m along each direction. All the cells can be reached by the fluid: they are considered as filled with a homogeneous facies (Facies #1). The only difference lies in the permeability values assigned to each cell. The permeability information is read from a CSV file.
This permeability field is considered through its log transform, by integer classes.
The permeability values vary between 1.086 and 26831.300. Its log transform varies between 0.083 and 10.197, and therefore in integer classes lying from 0 to 10.

In [ ]:
datcsv = pd.read_csv(
    gdoc.loadData("Fluid", "Permeability.csv", verbose=False), header=None
)
datnp = np.array(datcsv)
logdatnp = np.log(datnp)
grid = gl.DbGrid.create([50, 50])

nFacies = 1
nFluids = 1
iuid = grid.addColumnsByConstant(nFacies, 1.0, "Facies")

iuid = grid.addColumns(datnp, "Permeability")
iuid = grid.addColumns(logdatnp.astype(int), "LogPerm")

In [ ]:
dbfmt = gl.DbStringFormat.createFromFlags(
    False, False, False, True, names=["Permeability", "LogPerm"]
)
grid.display(dbfmt)

The log-permeability field is presented in the next figure.

In [ ]:
fig, ax = plt.subplots(1, 1)
ax.raster(grid, name="LogPerm", flagLegend=True)
plt.show()

The (single) fluid is provided by a single injection well, located in the center of the block system. For the Fluid Propagation procedure, we consider that the central cell (25,25) is filled with fluid.

In [ ]:
iuid = grid.addColumnsByConstant(nFluids, gl.TEST, "Fluid")
center = grid.indiceToRank([25, 25])
grid.setValue("Fluid", center, 1.0)

## Result

We let the fluid invade the whole block system using the Fluid Propagation algorithm. As the whole block system is constituted of a single facies and as a single fluid is considered, the invasion speed only relies upon the permeability field.

Then we let the Fluid Propagation algorithm run during several iterations (as one additional cell is populated in each iteration). At the end of the process, the (single) fluid invades the whole block system. Note that a cell with a zero log-permeability is still invaded, but very slowly...

So the only interesting map is the one corresponding to the date (rank of the iteration) when the cell is invaded by the fluid. This map is presented in the next figure:

In [ ]:
sl = 1
sm = 3
speeds = [sm, sm, sm, sm, sl, sl]
err = gl.fluidPropagation(
    grid, "Facies", "Fluid", "LogPerm", "", nFacies, nFluids, 1, speeds, verbose=True
)

We can check the consistency between the location of the permeability bareers and the slowliness of the invasion. This map can finally be used in order to derive the break through time at the four wells located in the corners of the block system.

In [ ]:
ax = gp.raster(grid, "Eden.Date", flagLegend=True)